In [ ]:
# CIFM-XAI reviewer revision. Run the entire file as ONE Colab Python cell.
# Set Runtime > Change runtime type > GPU before running the final experiment.
# Optionally set ZIP_PATH to an existing /content or mounted Google Drive file.
ZIP_PATH = "/content/Strategic_Identity_Fragmentation_CIFM.zip"
OUTPUT_PARENT = "/content"
SEEDS = (42, 127, 340, 541, 819)
SPLIT_SEED = 2026
EPOCHS = 3
MAX_LENGTH = 64
BATCH_SIZE = 16
BOOTSTRAPS = 2000
RANDOMIZATIONS = 5000

import os
import sys
import json
import hashlib
import itertools
import platform
import re
import shutil
import subprocess
import tempfile
import time
import unicodedata
import zipfile
from pathlib import Path
from collections import defaultdict
from importlib.metadata import version

if "google.colab" in sys.modules or Path("/content").exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "--only-binary=sentencepiece",
                           "transformers==4.51.3", "sentencepiece==0.2.1",
                           "xgboost==3.0.0", "psutil"])
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np
import pandas as pd
import psutil
from scipy.stats import wilcoxon
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

POSITIVE = ("message_clarity_score", "cultural_adaptation_score",
            "conflict_resolution_score", "politeness_score", "engagement_level",
            "active_listening_score", "context_understanding_score",
            "adaptation_speed", "communication_competence_level")
RISK = ("response_time_ms", "language_switch_frequency", "miscommunication_events")
COMM_FEATURES = ("conversation_turns", "average_message_length", "clarification_requests",
                 "feedback_response_quality", "cultural_reference_usage",
                 "communication_consistency")
VOTES = ("not offensive (no)", "slightly offensive (o1)", "somewhat offensive (o2)",
         "moderately offensive (o3)", "extremely offensive (o4)")
TRANSFORMERS = {"mBERT": "google-bert/bert-base-multilingual-cased",
                "XLM-R": "FacebookAI/xlm-roberta-base"}
ENCODER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# MiniLM pooling follows the archive's masked-token mean pooling.
PRIMARY = "CIFM-XAI"


def normalize(value):
    value = unicodedata.normalize("NFKC", str(value)).casefold()
    return " ".join(re.findall(r"[^\W_]+", value, flags=re.UNICODE))


def load_data(path):
    frames = []
    with zipfile.ZipFile(path) as archive:
        names = sorted(archive.namelist())
        for name in names:
            if "/data/raw/seegull/" not in name or not name.endswith(".csv"):
                continue
            task = Path(name).parent.name
            if task not in ("stereotypes", "offensiveness"):
                continue
            data = pd.read_csv(archive.open(name))
            data.columns = data.columns.str.strip().str.lower()
            if task == "stereotypes":
                counts = data[["non_stereotype", "stereotype"]].to_numpy(float)
                numerator = counts[:, 1]
                translated = data["translated_attribute_list"]
            else:
                counts = data[list(VOTES)].to_numpy(float)
                numerator = counts @ (np.arange(5) / 4)
                translated = data["translated attribute"]
            denominator = counts.sum(axis=1)
            score = np.divide(numerator, denominator, out=np.full(len(data), np.nan),
                              where=denominator > 0)
            locale = Path(name).stem
            frames.append(pd.DataFrame({
                "id": [f"{task}/{locale}/{i}" for i in range(len(data))],
                "task": task, "locale": locale,
                "text": data["attribute"].fillna("").astype(str).map(str.strip),
                "translation": translated.fillna("").astype(str),
                "score": score}))
        candidates = [n for n in names if n.endswith(
            "/data/raw/communication/cross_cultural_communication_dataset.csv")]
        if len(frames) != 46 or len(candidates) != 1:
            raise ValueError("Upload the original CIFM archive with all 46 SeeGULL CSVs and the Communication CSV.")
        comm = pd.read_csv(archive.open(candidates[0]))
    raw = pd.concat(frames, ignore_index=True)
    raw["text_key"] = raw.text.map(normalize)
    raw["translation_key"] = raw.translation.map(normalize)
    sg = raw.loc[raw.score.notna() & raw.text_key.ne("")].copy()
    # Aggregate repeated annotations for the same task, locale and normalized text.
    keys = ["task", "locale", "text_key"]
    aliases = sg[keys + ["translation_key"]].copy()
    sg = sg.groupby(keys, sort=False, as_index=False).agg(
        id=("id", "first"), text=("text", "first"), score=("score", "mean"))
    alias_index = aliases.merge(sg[keys].reset_index(), on=keys, validate="many_to_one")
    parent = list(range(len(sg)))

    def root(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    lookup = {}
    # Shared normalized originals OR shared normalized translations link records.
    for row in alias_index.itertuples(index=False):
        for key in (row.text_key, row.translation_key):
            if key:
                match = lookup.setdefault(key, row.index)
                parent[root(row.index)] = root(match)
    sg["group"] = [root(i) for i in range(len(sg))]
    comm["id"] = [f"COMM/{i}" for i in range(len(comm))]
    comm["group"] = pd.factorize(pd.MultiIndex.from_frame(comm[list(COMM_FEATURES)]))[0]
    audit = {"SeeGULL_raw": len(raw), "SeeGULL_valid": len(aliases),
             "SeeGULL_retained": len(sg), "Communication_raw": len(comm),
             "SeeGULL_groups": int(sg.group.nunique()),
             "largest_SeeGULL_group": int(sg.group.value_counts().max()),
             "communication_excluded_label_inputs": list(POSITIVE + RISK),
             "communication_allowed_predictors": list(COMM_FEATURES),
             "SeeGULL_allowed_predictor": "original attribute text only",
             "cross_dataset_transfer": "Not applicable: distinct inputs and proxy targets"}
    return sg, comm, audit


def make_split(groups):
    # Select whole groups to minimize deviations from 70/15/15 row proportions.
    # Candidate selection is based only on group sizes, prior to label construction.
    keys, indexes, counts = np.unique(groups, return_inverse=True, return_counts=True)
    best_error = np.inf
    for offset in range(256):
        dev, test = train_test_split(np.arange(len(keys)), test_size=.15,
                                     random_state=SPLIT_SEED + offset)
        train, val = train_test_split(dev, test_size=.15 / .85,
                                      random_state=SPLIT_SEED + offset)
        parts = (train, val, test)
        fractions = np.array([counts[p].sum() for p in parts]) / len(groups)
        error = np.abs(fractions - [.7, .15, .15]).sum()
        if error < best_error:
            best_error = error
            best_parts = parts
    sets = [np.flatnonzero(np.isin(indexes, part)) for part in best_parts]
    assert sum(map(len, sets)) == len(groups)
    assert all(not np.intersect1d(groups[a], groups[b]).size
               for a, b in itertools.combinations(sets, 2))
    return sets


def targets(name, data, train):
    if name == "SeeGULL":
        score = data.score.to_numpy()
    else:
        columns = list(POSITIVE + RISK)
        raw = data[columns].copy()
        # Use an explicit ordinal map in place of the archive's row-order encoding.
        mapping = {"Low": 0, "Moderate": 1, "High": 2, "Expert": 3}
        raw["communication_competence_level"] = raw["communication_competence_level"].map(mapping)
        if raw.isna().any().any():
            raise ValueError("Communication target inputs contain missing or unseen values.")
        raw = raw.to_numpy(float)
        scaler = MinMaxScaler(clip=True).fit(raw[train])
        components = scaler.transform(raw)
        components[:, :len(POSITIVE)] = 1 - components[:, :len(POSITIVE)]
        score = components.mean(axis=1)
    threshold = float(np.median(score[train]))
    return (score >= threshold).astype(int), threshold


def tune(y, score):
    precision, recall, thresholds = precision_recall_curve(y, score)
    values = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    index = int(np.argmax(values))
    return float(thresholds[index]), float(values[index])


def evaluate(y, score, threshold):
    pred = score >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"accuracy": accuracy_score(y, pred), "precision": tp / max(tp + fp, 1),
            "recall": tp / max(tp + fn, 1), "f1": f1_score(y, pred),
            "roc_auc": roc_auc_score(y, score), "tn": tn, "fp": fp, "fn": fn, "tp": tp}


def feature_views(raw, indexes, name, seed):
    a, b, c = indexes
    scaler = StandardScaler().fit(raw[a])
    scaled = [scaler.transform(raw[i]) for i in (a, b, c)]
    if name == "SeeGULL":
        reducer = TruncatedSVD(n_components=min(50, raw.shape[1] - 1), random_state=seed)
    else:
        reducer = PCA(n_components=min(10, raw.shape[1]), random_state=seed)
    reducer.fit(scaled[0])
    base = [reducer.transform(part) for part in scaled]
    iso = IsolationForest(n_estimators=60, random_state=seed, n_jobs=2).fit(base[0])
    def augment(features):
        return np.column_stack([features, -iso.score_samples(features)])
    return base, augment, [augment(part) for part in base]


def classic_models(seed):
    from xgboost import XGBClassifier
    return {
        "LR": LogisticRegression(max_iter=1000, random_state=seed),
        "RF": RandomForestClassifier(n_estimators=150, min_samples_leaf=3, random_state=seed, n_jobs=2),
        "ET": ExtraTreesClassifier(n_estimators=150, min_samples_leaf=3, random_state=seed, n_jobs=2),
        "XGBoost": XGBClassifier(n_estimators=150, max_depth=4, learning_rate=.08,
                                 random_state=seed, n_jobs=2, eval_metric="logloss")}


def seed_torch(seed):
    import torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)


def tokenize(tokenizer, texts):
    # Pretrained tokenization has no fitted corpus statistics.
    tokenizer.padding_side = "right"
    return tokenizer(list(texts), padding="max_length", truncation=True,
                     max_length=MAX_LENGTH, return_tensors="pt")


def mini_embeddings(texts, device):
    import torch
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(ENCODER)
    model = AutoModel.from_pretrained(ENCODER).to(device).eval()
    unique, idx = np.unique(np.asarray(texts, dtype=str), return_inverse=True)
    tokens = tokenize(tok, unique)
    output = []
    with torch.inference_mode():
        for start in range(0, len(unique), 64):
            batch = {key: value[start:start + 64].to(device) for key, value in tokens.items()}
            hidden = model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1)
            output.append(((hidden * mask).sum(1) / mask.sum(1).clamp_min(1)).cpu().numpy())
    return np.vstack(output)[idx], model.config._commit_hash


def transfer_batch(rows, keys, device, trim_padding):
    if trim_padding:
        # Tokenizers right-pad these models. Keep all real tokens and round the
        # tensor width to a multiple of 8 when possible.
        mask = rows[keys.index("attention_mask")]
        width = min(mask.shape[1], max(8, (int(mask.sum(1).max()) + 7) // 8 * 8))
        rows = tuple(
            tensor[:, :width] if key in ("input_ids", "attention_mask", "token_type_ids")
            else tensor for key, tensor in zip(keys, rows[:-1])
        ) + (rows[-1],)
    return [tensor.to(device, non_blocking=device == "cuda") for tensor in rows]


def neural(name, inputs, y, indexes, seed, device):
    import gc
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    seed_torch(seed)
    a, b, c = indexes
    dataset_keys = list(inputs)

    class LSTM(nn.Module):
        def __init__(self, token_input):
            super().__init__()
            self.embedding = nn.Embedding(120000, 48, padding_idx=0) if token_input else None
            self.recurrent = nn.LSTM(48 if token_input else 1, 48, batch_first=True)
            self.out = nn.Linear(48, 2)

        def forward(self, **batch):
            x = batch["input_ids"]
            if self.embedding is not None:
                embedding = self.embedding(x)
                last = batch["attention_mask"].sum(1).clamp_min(1) - 1
            else:
                embedding = x.unsqueeze(-1)
                last = torch.full((len(x),), x.shape[1] - 1, device=x.device)
            sequence, _ = self.recurrent(embedding)
            return self.out(sequence[torch.arange(len(x), device=x.device), last])

    revision = None
    if name == "LSTM":
        model = LSTM("attention_mask" in inputs)
        learning_rate = .001
    else:
        from transformers import AutoModelForSequenceClassification
        model = AutoModelForSequenceClassification.from_pretrained(
            TRANSFORMERS[name], num_labels=2)
        revision = model.config._commit_hash
        learning_rate = 2e-5
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    amp = device == "cuda" and name != "LSTM"
    trim_padding = name != "LSTM" and "attention_mask" in inputs
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    criterion = nn.CrossEntropyLoss()
    labels = torch.as_tensor(y, dtype=torch.long)

    def batches(index, training):
        data = TensorDataset(*(inputs[key][index] for key in dataset_keys), labels[index])
        return DataLoader(data, batch_size=BATCH_SIZE if name != "LSTM" else 128,
                          shuffle=training, pin_memory=device == "cuda",
                          generator=torch.Generator().manual_seed(seed))

    def logits(batch):
        value = model(**dict(zip(dataset_keys, batch[:-1])))
        return value if name == "LSTM" else value.logits

    epochs = EPOCHS if name != "LSTM" else 8
    started = time.monotonic()
    for epoch in range(1, epochs + 1):
        model.train()
        loader = batches(a, True)
        steps = len(loader)
        for step, rows in enumerate(loader, 1):
            rows = transfer_batch(rows, dataset_keys, device, trim_padding)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp):
                loss = criterion(logits(rows), rows[-1])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
            scaler.step(optimizer)
            scaler.update()
            if step == 1 or step % 100 == 0 or step == steps:
                if not torch.isfinite(loss).item():
                    raise FloatingPointError(f"{name}: non-finite training loss")
                done = (epoch - 1) * steps + step
                elapsed = time.monotonic() - started
                eta = elapsed * (epochs * steps - done) / done
                print(f"  {name} | epoch {epoch}/{epochs} | batch {step}/{steps} "
                      f"| training ETA ~{eta / 60:.1f} min", flush=True)

    scores = []
    model.eval()
    with torch.inference_mode():
        for index in (b, c):
            probabilities = []
            for rows in batches(index, False):
                rows = transfer_batch(rows, dataset_keys, device, trim_padding)
                with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp):
                    prediction = logits(rows)
                probabilities.append(prediction.float().softmax(1)[:, 1].cpu().numpy())
            scores.append(np.concatenate(probabilities))
    print(f"  {name} | fit and evaluation finished in "
          f"{(time.monotonic() - started) / 60:.1f} min", flush=True)
    del model, optimizer
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return scores, revision


def fit_classic(raw, indexes, y, name, seed):
    train, val, test = indexes
    base, _, augmented = feature_views(raw, indexes, name, seed)

    def fitted_probabilities(views):
        models = classic_models(seed)
        for model in models.values():
            model.fit(views[0], y[train])
        return models, {key: [m.predict_proba(part)[:, 1] for part in views[1:]]
                        for key, m in models.items()}

    _, scores = fitted_probabilities(base)
    models, cifm_scores = fitted_probabilities(augmented)
    # w_k = (F1_k + epsilon) / sum_j(F1_j + epsilon), all F1 values from validation.
    weights = np.array([tune(y[val], pair[0])[1] for pair in cifm_scores.values()]) + 1e-8
    weights /= weights.sum()
    scores[PRIMARY] = [
        weights @ np.stack([pair[partition] for pair in cifm_scores.values()])
        for partition in (0, 1)]
    base_weights = np.array([tune(y[val], scores[key][0])[1] for key in models]) + 1e-8
    base_weights /= base_weights.sum()
    scores["CIFM without anomaly"] = [
        base_weights @ np.stack([scores[key][partition] for key in models])
        for partition in (0, 1)]
    scores["Isolation Forest"] = [part[:, -1] for part in augmented[1:]]
    scores["Majority"] = [np.full(len(i), float(y[train].mean() >= .5))
                          for i in (val, test)]
    threshold = tune(y[val], scores[PRIMARY][0])[0]
    importances = permutation_importance(models, weights, augmented[2], y[test],
                                         threshold, seed, name)
    return scores, list(base), dict(zip(models, weights.tolist())), importances


def permutation_importance(models, weights, features, y, threshold, seed, name):
    # Binary F1 on the held-out test partition after all model choices are frozen.
    # Each projected feature is permuted independently, including the anomaly score.
    if seed != SEEDS[0]:
        return []
    rng = np.random.default_rng(seed)

    def score(matrix):
        probs = weights @ np.stack([m.predict_proba(matrix)[:, 1] for m in models.values()])
        return f1_score(y, probs >= threshold, zero_division=0)

    original = score(features)
    report = []
    for column in range(features.shape[1]):
        permuted = features.copy()
        drops = []
        for _ in range(10):
            permuted[:, column] = rng.permutation(features[:, column])
            drops.append(original - score(permuted))
        kind = "SVD" if name == "SeeGULL" else "PCA"
        feature = f"{kind}_{column + 1}" if column + 1 < features.shape[1] else "anomaly_score"
        report.append({"dataset": name, "feature": feature, "seed": seed,
                       "mean_test_f1_drop": np.mean(drops), "sd": np.std(drops, ddof=1)})
    return report


def group_intervals(predictions, seed):
    # Paired test-group bootstrap. Resample whole text/duplicate groups and apply
    # identical draws to all models. CIs condition on the split and trained runs.
    rng = np.random.default_rng(seed)
    rows = []
    for dataset, data in predictions.groupby("dataset"):
        models = sorted(data.model.unique())
        test = data[data.model.eq(PRIMARY) & data.seed.eq(SEEDS[0])]
        group_index = pd.Index(test.group.unique())
        matrices = {}
        for (name, run), records in data.groupby(["model", "seed"]):
            codes = group_index.get_indexer(records.group)
            assert (codes >= 0).all() and len(records) == len(test)
            counts = np.zeros((len(group_index), 3), dtype=np.int32)
            y, p = records.y.to_numpy(bool), records.pred.to_numpy(bool)
            for i, mask in enumerate((y & p, ~y & p, y & ~p)):
                np.add.at(counts[:, i], codes, mask)
            matrices[name, run] = counts
        stacked = np.stack([matrices[m, run] for m in models for run in SEEDS], axis=1)
        draws = []
        for _ in range(BOOTSTRAPS):
            chosen = rng.integers(len(group_index), size=len(group_index))
            total = stacked[chosen].sum(axis=0).reshape(len(models), len(SEEDS), 3)
            tp, fp, fn = total.transpose(2, 0, 1)
            f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
            draws.append(f1.mean(axis=1))
        draws = np.asarray(draws)
        primary = draws[:, models.index(PRIMARY)]
        for i, model in enumerate(models):
            low, high = np.quantile(draws[:, i], [.025, .975])
            gap_low, gap_high = np.quantile(primary - draws[:, i], [.025, .975])
            p_value = np.nan
            if model != PRIMARY:
                left = stacked[:, models.index(PRIMARY) * len(SEEDS):
                               (models.index(PRIMARY) + 1) * len(SEEDS)]
                right = stacked[:, i * len(SEEDS):(i + 1) * len(SEEDS)]
                p_value = paired_group_randomization(left, right, rng)
            rows.append({"dataset": dataset, "model": model, "f1_ci_lower": low,
                         "f1_ci_upper": high, "cifm_minus_model_ci_lower": gap_low,
                         "cifm_minus_model_ci_upper": gap_high,
                         "paired_group_randomization_p": p_value})
    result = pd.DataFrame(rows)
    result["paired_group_holm_p"] = holm_adjust(result.paired_group_randomization_p)
    return result


def paired_group_randomization(left, right, rng):
    # Swap complete groups between models across all seeded fits together.
    def f1(counts):
        tp, fp, fn = counts.transpose(2, 0, 1)
        return (2 * tp / np.maximum(2 * tp + fp + fn, 1)).mean(axis=1)

    lhs = left.sum(axis=0).reshape(1, len(SEEDS), 3)
    rhs = right.sum(axis=0).reshape(1, len(SEEDS), 3)
    delta = (right - left).reshape(len(left), -1).astype(float)
    observed = abs(float((f1(lhs) - f1(rhs))[0]))
    exceed = 0
    for start in range(0, RANDOMIZATIONS, 128):
        draws = min(128, RANDOMIZATIONS - start)
        swaps = rng.integers(0, 2, size=(draws, len(left)), dtype=np.int8)
        movement = (swaps @ delta).reshape(draws, len(SEEDS), 3)
        gap = f1(lhs + movement) - f1(rhs - movement)
        exceed += int((np.abs(gap) >= observed - 1e-12).sum())
    return (exceed + 1) / (RANDOMIZATIONS + 1)


def holm_adjust(values):
    ordered = values.dropna().sort_values()
    adjusted = np.minimum(1, np.maximum.accumulate(
        ordered.to_numpy() * np.arange(len(ordered), 0, -1)))
    return pd.Series(adjusted, index=ordered.index).reindex(values.index)


def statistical_tests(metrics):
    rows = []
    for dataset, data in metrics.groupby("dataset"):
        wide = data.pivot(index="seed", columns="model", values="f1")
        for model in wide.columns.drop(PRIMARY):
            delta = wide[PRIMARY] - wide[model]
            p = 1.0 if np.allclose(delta, 0) else wilcoxon(
                delta, alternative="two-sided", method="auto").pvalue
            rows.append({"dataset": dataset, "baseline": model,
                         "mean_cifm_minus_baseline_f1": delta.mean(),
                         "n_paired_seeds": len(delta), "wilcoxon_p": p})
    result = pd.DataFrame(rows).sort_values("wilcoxon_p").reset_index(drop=True)
    result["holm_adjusted_p"] = holm_adjust(result.wilcoxon_p)
    return result


def save_plots(metrics, output):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import math
    plt.rcParams.update({"font.size": 9})
    for dataset, subset in metrics.groupby("dataset"):
        first = subset[subset.seed.eq(SEEDS[0])]
        fig, axes = plt.subplots(math.ceil(len(first) / 3), 3, squeeze=False,
                                 figsize=(9, 2.6 * math.ceil(len(first) / 3)))
        for axis in axes.flat:
            axis.set_visible(False)
        for axis, row in zip(axes.flat, first.itertuples(index=False)):
            axis.set_visible(True)
            matrix = np.array([[row.tn, row.fp], [row.fn, row.tp]])
            axis.imshow(matrix, cmap="Blues")
            for i, j in itertools.product(range(2), repeat=2):
                shade = "white" if matrix[i, j] > matrix.max() * .6 else "black"
                axis.text(j, i, f"{matrix[i, j]:,}", ha="center", va="center", color=shade)
            axis.set(title=row.model, xlabel="Prediction", ylabel="Proxy label",
                     xticks=[0, 1], yticks=[0, 1])
        fig.suptitle(f"{dataset} held-out confusion matrices, seed {SEEDS[0]}")
        fig.tight_layout()
        fig.savefig(output / f"confusion_{dataset}.png", dpi=300)
        plt.close(fig)
        summary = subset.groupby("model").f1.agg(["mean", "std"]).sort_values("mean")
        fig, axis = plt.subplots(figsize=(7, 5))
        axis.barh(summary.index, summary["mean"], xerr=summary["std"],
                  capsize=3, color="#376783")
        axis.set(xlabel="Test F1 mean ± SD across seeds", xlim=(0, 1.05), title=dataset)
        fig.tight_layout()
        fig.savefig(output / f"f1_{dataset}.png", dpi=300)
        plt.close(fig)


def run(zip_path):
    import torch
    from transformers import AutoTokenizer
    output = Path(tempfile.mkdtemp(prefix="CIFM_XAI_results_", dir=OUTPUT_PARENT))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("CPU selected. GPU is recommended for the full 10-fit transformer comparison.", flush=True)
    else:
        print(f"GPU selected: {torch.cuda.get_device_name()} | Transformer AMP: FP16", flush=True)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"{zip_path.name} is empty or invalid. Upload the original "
                         "Strategic_Identity_Fragmentation_CIFM.zip.")
    sg, comm, audit = load_data(zip_path)
    (output / "data_audit.json").write_text(json.dumps(audit, indent=2))
    cpu_info = Path("/proc/cpuinfo")
    cpu_model = re.search(r"^model name\s*:\s*(.+)$", cpu_info.read_text(), re.M) if cpu_info.exists() else None
    with zip_path.open("rb") as source:
        archive_hash = hashlib.file_digest(source, "sha256").hexdigest()
    metadata = {"cpu": cpu_model[1] if cpu_model else platform.processor(),
                "physical_cores": psutil.cpu_count(logical=False),
                "logical_cores": psutil.cpu_count(logical=True),
                "memory_gib": round(psutil.virtual_memory().total / 2**30, 3),
                "gpu": torch.cuda.get_device_name() if device == "cuda" else None,
                "python": platform.python_version(),
                "packages": {p: version(p) for p in ("torch", "transformers", "xgboost",
                             "scikit-learn", "scipy", "numpy", "pandas", "matplotlib")},
                "seed_values": list(SEEDS), "split_seed": SPLIT_SEED, "epochs": EPOCHS,
                "max_length": MAX_LENGTH, "transformer_batch_size": BATCH_SIZE,
                "pytorch_device": device, "encoder": ENCODER,
                "transformer_precision": "fp16_amp" if device == "cuda" else "fp32",
                "transformer_batch_padding": "trim right padding to batch width, rounded to 8",
                "transformers": TRANSFORMERS, "model_revisions": {},
                "zip_sha256": archive_hash}
    if "google.colab" in sys.modules:
        shell = get_ipython()
        # Export the exact submitted cell with the run when a history entry is available.
        cells = [cell for cell in shell.history_manager.input_hist_raw
                 if "def run(zip_path):" in cell and "def load_data(path):" in cell]
        if cells:
            (output / "reviewer_revision_code.py").write_text(cells[-1])
    results, records, splits, manifests, importance, phase_rows = [], [], [], [], [], []
    for dataset, data in (("SeeGULL", sg), ("Communication", comm)):
        indexes = make_split(data.group.to_numpy())
        y, label_threshold = targets(dataset, data, indexes[0])
        for phase, index in zip(("train", "validation", "test"), indexes):
            if len(np.unique(y[index])) != 2:
                raise ValueError(f"{dataset} {phase} has only one class.")
            splits.append({"dataset": dataset, "partition": phase, "n": len(index),
                           "share": len(index) / len(data), "positive_n": y[index].sum(),
                           "groups": data.group.iloc[index].nunique(),
                           "train_target_threshold": label_threshold})
            manifests.append(pd.DataFrame({"dataset": dataset, "partition": phase,
                "id": data.id.iloc[index].to_numpy(), "group": data.group.iloc[index].to_numpy(),
                "y": y[index]}))
        pd.DataFrame(splits).to_csv(output / "split_summary.csv", index=False)
        pd.concat(manifests).to_csv(output / "split_manifest.csv", index=False)
        tokenized = {}
        if dataset == "SeeGULL":
            raw, revision = mini_embeddings(data.text, device)
            metadata["model_revisions"][ENCODER] = revision
            for key, repo in TRANSFORMERS.items():
                tokenizer = AutoTokenizer.from_pretrained(repo)
                tokenized[key] = dict(tokenize(tokenizer, data.text))
            lstm_tok = AutoTokenizer.from_pretrained(TRANSFORMERS["mBERT"])
            lstm_tokens = dict(tokenize(lstm_tok, data.text))
            lstm_tokens.pop("token_type_ids", None)
        else:
            raw = data[list(COMM_FEATURES)].to_numpy(float)
        train, val, test = indexes
        for seed in SEEDS:
            print(f"{dataset} | seed {seed} | {device}", flush=True)
            scores, base, weights, drops = fit_classic(raw, indexes, y, dataset, seed)
            importance.extend(drops)
            if dataset == "SeeGULL":
                scores["LSTM"], _ = neural("LSTM", lstm_tokens, y, indexes, seed, device)
                for key in TRANSFORMERS:
                    print(f"  Fine-tuning {key}", flush=True)
                    scores[key], revision = neural(key, tokenized[key], y, indexes, seed, device)
                    metadata["model_revisions"][key] = revision
            else:
                # Fixed-order numeric feature sequence, with no temporal interpretation.
                values = np.empty((len(data), base[0].shape[1]), dtype=np.float32)
                for index, view in zip(indexes, base):
                    values[index] = view
                scores["LSTM"], _ = neural("LSTM", {"input_ids": torch.from_numpy(values)},
                                          y, indexes, seed, device)
            for model, (val_score, test_score) in scores.items():
                threshold, validation_f1 = tune(y[val], val_score)
                if model == "Majority":
                    threshold = .5
                    validation_f1 = f1_score(y[val], val_score >= threshold, zero_division=0)
                results.append({"dataset": dataset, "seed": seed, "model": model,
                                "threshold": threshold, "validation_f1": validation_f1,
                                **evaluate(y[test], test_score, threshold)})
                records.append(pd.DataFrame({"dataset": dataset, "seed": seed, "model": model,
                    "id": data.id.iloc[test].to_numpy(), "group": data.group.iloc[test].to_numpy(),
                    "y": y[test], "score": test_score, "pred": (test_score >= threshold).astype(int)}))
            phase_rows.append({"dataset": dataset, "seed": seed, **weights})
            pd.DataFrame(results).to_csv(output / "metrics_by_seed.csv", index=False)
            pd.DataFrame(phase_rows).to_csv(output / "ensemble_weights.csv", index=False)
            pd.DataFrame(importance).to_csv(output / "permutation_importance.csv", index=False)
            (output / "environment.json").write_text(json.dumps(metadata, indent=2))
    metrics = pd.DataFrame(results)
    predictions = pd.concat(records, ignore_index=True)
    predictions.to_csv(output / "test_predictions.csv.gz", index=False)
    summary = metrics.groupby(["dataset", "model"])[
        ["accuracy", "precision", "recall", "f1", "roc_auc"]].agg(["mean", "std"])
    summary.columns = ["_".join(col) for col in summary.columns]
    summary.reset_index().merge(group_intervals(predictions, SPLIT_SEED),
        on=["dataset", "model"], validate="one_to_one").to_csv(output / "results_summary.csv", index=False)
    statistical_tests(metrics).to_csv(output / "paired_seed_tests.csv", index=False)
    metrics.loc[metrics.precision.eq(1), ["dataset", "seed", "model", "tp", "fp", "fn", "tn"]
                ].to_csv(output / "perfect_precision_audit.csv", index=False)
    save_plots(metrics, output)
    (output / "interpretation.txt").write_text(
        "Both endpoints are score-derived proxies. SeeGULL derives scores from votes and "
        "uses original-language text only. Communication derives scores from 12 fields "
        "excluded from prediction, including competence encoded in ordinal order. "
        "Transformer classifiers are applicable to text-bearing SeeGULL only. Both datasets "
        "are modeled separately. This study design provides no direct cross-domain transfer test. "
        "Splits target 70/15/15 at the row level while holding normalized text/translation or "
        "duplicate predictor groups together. Text normalization does not detect all paraphrases. "
        "A single held-out test partition per dataset is fixed across all seeds. Standard deviations "
        "measure variability across seeded fits on that split. Paired Wilcoxon tests across seeds "
        "are exploratory and conditional on the same split; they do not establish population-level "
        "superiority. F1 confidence intervals use paired test-group bootstrap averaged over seeds. "
        "Test-group randomization p-values assess exchangeability of paired predictions on the "
        "held-out partition. Holm adjustment covers all dataset-baseline comparisons. "
        "Permutation importance is binary-F1 reduction on held-out test features for the first seed, "
        "after model selection, averaged over ten permutations. Feature effects are measured in "
        "the projected feature space. Neither the scores nor the importance estimates establish "
        "construct validity or causal relationships.\n")
    bundle = shutil.make_archive(str(output), "zip", root_dir=output)
    print(pd.read_csv(output / "results_summary.csv")[
        ["dataset", "model", "f1_mean", "f1_std", "roc_auc_mean"]].to_string(index=False))
    print(f"Finished. Download {bundle}", flush=True)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(bundle)
    return Path(bundle)


if __name__ == "__main__":
    project = Path(ZIP_PATH)
    if not project.is_file():
        from google.colab import files
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
        if len(candidates) != 1:
            raise ValueError("Upload exactly one original CIFM project ZIP.")
        project = candidates[0]
        del uploaded
    run(project)


Saving Strategic_Identity_Fragmentation_CIFM.zip to Strategic_Identity_Fragmentation_CIFM.zip
GPU selected: Tesla T4 | Transformer AMP: FP16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (32 outputs left out)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  XLM-R | epoch 1/3 | batch 1/1512 | training ETA ~17.5 min
  XLM-R | epoch 1/3 | batch 100/1512 | training ETA ~13.0 min
  XLM-R | epoch 1/3 | batch 200/1512 | training ETA ~12.8 min
  XLM-R | epoch 1/3 | batch 300/1512 | training ETA ~12.6 min
  XLM-R | epoch 1/3 | batch 400/1512 | training ETA ~12.3 min
  XLM-R | epoch 1/3 | batch 500/1512 | training ETA ~12.0 min
  XLM-R | epoch 1/3 | batch 600/1512 | training ETA ~11.7 min
  XLM-R | epoch 1/3 | batch 700/1512 | training ETA ~11.4 min
  XLM-R | epoch 1/3 | batch 800/1512 | training ETA ~11.2 min
  XLM-R | epoch 1/3 | batch 900/1512 | training ETA ~10.9 min
  XLM-R | epoch 1/3 | batch 1000/1512 | training ETA ~10.6 min
  XLM-R | epoch 1/3 | batch 1100/1512 | training ETA ~10.3 min
  XLM-R | epoch 1/3 | batch 1200/1512 | training ETA ~10.0 min
  XLM-R | epoch 1/3 | batch 1300/1512 | training ETA ~9.7 min
  XLM-R | epoch 1/3 | batch 1400/1512 | training ETA ~9.4 min
  XLM-R | epoch 1/3 | batch 1500/1512 | training ETA ~9.1 min
  XLM-R

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path
import shutil
import zipfile
from google.colab import files

root = Path("/content")
run_name = "CIFM_XAI_results_qa4f1hqf"
zip_path = root / f"{run_name}.zip"
results_dir = root / run_name

if zip_path.is_file() and zipfile.is_zipfile(zip_path):
    print(f"Existing ZIP mili: {zip_path.name}")
    files.download(str(zip_path))

elif (results_dir / "results_summary.csv").is_file():
    zip_path = shutil.make_archive(
        str(root / "CIFM_XAI_All_Outputs"), "zip", root_dir=results_dir
    )
    print(f"Results ZIP ban gayi: {zip_path}")
    files.download(zip_path)

else:
    print("Is Colab session mein original results ZIP ya completed folder nahi mila.")

Is Colab session mein original results ZIP ya completed folder nahi mila.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

# Prefer the exact run shown in your log.
results_dir = Path("/content/CIFM_XAI_results_qa4f1hqf")

# If that folder is unavailable, find the latest completed run.
if not (results_dir / "results_summary.csv").is_file():
    completed_runs = [
        folder for folder in Path("/content").glob("CIFM_XAI_results_*")
        if folder.is_dir() and (folder / "results_summary.csv").is_file()
    ]
    if not completed_runs:
        raise FileNotFoundError(
            "Results folder nahi mila. Ye cell usi Colab runtime mein run "
            "karein jahan training complete hui thi."
        )
    results_dir = max(
        completed_runs,
        key=lambda folder: (folder / "results_summary.csv").stat().st_mtime,
    )

zip_path = shutil.make_archive(
    "/content/CIFM_XAI_All_Outputs",
    "zip",
    root_dir=results_dir,
)

print(f"Run folder: {results_dir.name}")
print(f"ZIP saved: {zip_path}")
files.download(zip_path)

FileNotFoundError: Results folder nahi mila. Ye cell usi Colab runtime mein run karein jahan training complete hui thi.